# Sentiment Analysis with Logistic Regression

In this class you will work with a logistic regression for sentiment analysis. The code provide a low performance. After going through the code you will think of ways of improving it.

A quick recap of Logistic Regression: https://ml-cheatsheet.readthedocs.io/en/latest/logistic_regression.html#binary-logistic-regression


If you don't have Pytorch installed in your local machine, please follow these instructions:

https://pytorch.org/get-started/locally/


You can use the command "nvidia-smi", and "nvcc --version" to check the CUDA version that is installed on your machine.
Colab already ships with the pre-compiled dependencies, so it is very easy to use GPUS

In [ ]:
# Load and install required libraries
!pip install tqdm # We will need tqdm to generate pretty progress bars
!pip install nltk
import os
import string
import nltk
nltk.download('stopwords')

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import torch
import torch.nn as nn
import torch.optim as optim
from torchtext import data
from tqdm import tqdm, trange
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# Download data
# Download files from https://www.kaggle.com/columbine/imdb-dataset-sentiment-analysis-in-csv-format/download
# Upload to your Colab if needed, and unzip it with the following command
!mkdir sentiment-analysis-data
!unzip archive.zip -d sentiment-analysis-data
import os
input_dir_path = '/content/sentiment-analysis-data'
for dirname, _, filenames in os.walk(input_dir_path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Archive:  archive.zip
  inflating: sentiment-analysis-data/Test.csv  
  inflating: sentiment-analysis-data/Train.csv  
  inflating: sentiment-analysis-data/Valid.csv  
/content/sentiment-analysis-data/Train.csv
/content/sentiment-analysis-data/Valid.csv
/content/sentiment-analysis-data/Test.csv


In [ ]:
# Data inspection
!head -10 /content/sentiment-analysis-data/Train.csv

﻿text,label
"I grew up (b. 1965) watching and loving the Thunderbirds. All my mates at school watched. We played ""Thunderbirds"" before school, during lunch and after school. We all wanted to be Virgil or Scott. No one wanted to be Alan. Counting down from 5 became an art form. I took my children to see the movie hoping they would get a glimpse of what I loved as a child. How bitterly disappointing. The only high point was the snappy theme tune. Not that it could compare with the original score of the Thunderbirds. Thankfully early Saturday mornings one television channel still plays reruns of the series Gerry Anderson and his wife created. Jonatha Frakes should hand in his directors chair, his version was completely hopeless. A waste of film. Utter rubbish. A CGI remake may be acceptable but replacing marionettes with Homo sapiens subsp. sapiens was a huge error of judgment.",0
"When I put this movie in my DVD player, and sat down with a coke and some chips, I had some expectations. 

### Data processing

The following cell is parsing the data contained in the csv files that you just downloaded.
Despite the extensive code block there are a number of familiar steps happening here:


1.   Clean and tokenize the text to generate TF-IDF features;
2.   Pair the text with its relevance judgements (labels);
3.   Create Datasets and Iterators to work with Pytorch

In [ ]:
# Set the number of "input dimensions", this is the maximum sentence size that
# we will consider for sentiment analysis
# TODO you will need to tweak this number
input_dim = 500
batch_size = 32

# Load and tokenize CSV files to memory
def load_file(filepath, device, MAX_VOCAB_SIZE = 25_000):
    # Step 1 - Data preprocessing
    # Removing punctuation & split sentences.
    tokenizer = lambda x: str(x).translate(str.maketrans('', '', string.punctuation)).strip().split()

    # Use torchtext to create data fields for the text and labels
    text_data_field = data.Field(sequential=True, lower=True, tokenize=tokenizer, fix_length=input_dim, stop_words=stopwords.words('english'))
    label_data_field = data.Field(sequential=False, use_vocab=False)

    print("Loading from csv...")
    tv_datafields = [("text", text_data_field), ("label", label_data_field)]

    # Step 2 - Build Pytorch Dataset.
    # This Dataset is an abstraction to pack our data into a datastructure that
    # can be easily be manipulated by Pytorch
    train_dataset, valid_dataset, test_dataset = data.TabularDataset.splits(path=filepath,
                                                    train="Train.csv", validation="Valid.csv",
                                                    test="Test.csv", format="csv",
                                                    skip_header=True, fields=tv_datafields)
    print(train_dataset[0].__dict__.keys())


    # Step 3 - We will build our vocabulary based on the given text field that we created on step 1
    # We use a maximum threshold to save memory
    text_data_field.build_vocab(train_dataset, max_size = MAX_VOCAB_SIZE)
    print("Text vocabulary built.")

    # Step 4 - Build our dataset iterators to go through the data.
    train_iter = data.Iterator(train_dataset, device=device, train=True, batch_size=batch_size, shuffle=True)
    valid_iter = data.Iterator(valid_dataset, device=device, batch_size=batch_size, shuffle=True)
    test_iter = data.Iterator(test_dataset, device=device, batch_size=batch_size, shuffle=True)
    print("Iterator built.")
    return text_data_field, label_data_field, train_dataset, valid_dataset, test_dataset, train_iter, valid_iter, test_iter

# This will obtain the devices available for training/inference
# In your local machines if you have CUDA installed then you will be able to use them
# If not, Pytorch will use the CPU to train the model. CUDA comes pre-packed in Colab
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("I'm using this device: {}".format(device))
print("Don't forget to change the runtime type to GPU if you're working on Colab.")

max_vocab_size = 25000
text_data_field, label_data_field, train_dataset, valid_dataset, test_dataset, train_iter, valid_iter, test_iter = \
load_file('/content/sentiment-analysis-data', device, MAX_VOCAB_SIZE=max_vocab_size)

I'm using this device: cpu
Don't forget to change the runtime type to GPU if you're working on Colab.
Loading from csv...
dict_keys(['text', 'label'])
Text vocabulary built.
Iterator built.


### Exploring our data

In [ ]:
# Let's explore the words that will be used as features for our model
# 1 - Using the Vocab structure that was created with our text_data_field obtain the following:

# Hint: https://torchtext.readthedocs.io/en/latest/vocab.html
# These 3 variables may help you
freqs = text_data_field.vocab.freqs
stoi = text_data_field.vocab.stoi
itos = text_data_field.vocab.itos

# Question 1.1 - What's the 15 words that occur more frequently in our dataset?
# Another Hint: freqs is a collections.Counter object https://docs.python.org/3.7/library/collections.html#collections.Counter
print(YOUR CODE HERE)

# Question 1.2 - How many times does the 51th most frequent word occurs:
print(YOUR CODE HERE)

### Building a Logistic Regression model with Pytorch


In [ ]:
# We will use a logistic regression, which is a binary classifier to determine if the text reviews are positive or negative
class LogisticRegression(nn.Module):
  def __init__(self, input_dim, output_dim):
    super(LogisticRegression, self).__init__()
    self.linear = torch.nn.Linear(input_dim, output_dim)
    self.activation_functions = [nn.Sigmoid(), nn.Softmax(),
                                 nn.LogSigmoid(), nn.LogSoftmax(),
                                 nn.ReLU(), nn.Tanh()]
    # TODO change the activation function here
    self.act_func = self.activation_functions[0]

  def forward(self, x):
    # Return the application of the linear layer over the input vector x
    logits = self.linear(x.T)
    probs = self.act_func(logits)
    # Squeeze values to supress deprecation warnings due to different vector shapes
    return logits.T.squeeze(0), probs.T.squeeze(0)

  # Evaluating our Model with accuracy
  def binary_accuracy(self, preds, y, act_fct):
      # Return accuracy per batch
      # rounded_preds = torch.round(act_fct(preds))
      correct = (preds.int() == y.int()).float()
      acc = correct.sum() / len(correct)
      return acc

### Preparing our model to train and evaluate

Also part of our Machine Learning recipe we need a loss function (To evaluate by how much an example fails), and an optimizer to update our network weights, and reduce our losses.

In [ ]:
# Initialize required variables
batch_size = 100 # Number of samples used to compute the gradients in each iteration
epochs = 5
output_dim = 1
learning_rate = 0.001

# This input dim was determined upon data processing time
model = LogisticRegression(input_dim, output_dim)

# Use binary cross entropy loss for binary classification
loss_func = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Let's send our model and loss function to the device. Hopefully it is a GPU
model = model.to(device)
loss_func = loss_func.to(device)

### The training and evaluation loop

In [ ]:
# Let's create our training and eval functions
def train(model, iterator, loss_func, optimizer):
  epoch_loss = 0.0
  epoch_acc = 0.0

  # Signal Pytorch that we are training our model
  model.train()

  total_steps = len(iterator)
  # for i, batch in enumerate(tqdm(iterator, total=total_steps, desc='Train - Iteration')):
  for i, batch in enumerate(iterator):
    optimizer.zero_grad()

    # This will invoke our logistic regression model
    logits, predictions = model(batch.text.float())

    loss = loss_func(predictions, batch.label.float())
    acc = model.binary_accuracy(predictions, batch.label, model.act_func)

    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()
    epoch_acc += acc.item()
    if i % 100 == 0:
      # Print loss update every 100 batches/steps
      print(f"[{i}/{total_steps}] : Loss: {loss.item():.2f}")
  return epoch_loss / total_steps, epoch_acc / total_steps


def evaluate(model, iterator, loss_func):
  epoch_loss = 0.0
  epoch_acc = 0.0

  # Signal pytorch that we are evaluating our model
  model.eval()

  total_steps = len(iterator)
  # Use torch no grad to signal that we do not want to update our gradients
  with torch.no_grad():
    #for i, batch in enumerate(tqdm(iterator, total=total_steps, desc='Eval - Iteration')):
    for i, batch in enumerate(iterator):
      # prediction [batch_size]
      logits, predictions = model(batch.text.float())

      loss = loss_func(predictions, batch.label.float())
      # Let's use the function that we defined earlier
      acc = model.binary_accuracy(predictions, batch.label, model.act_func)

      epoch_loss += loss.item()
      epoch_acc += acc.item()

  return epoch_loss / total_steps,  epoch_acc / total_steps

### Putting everything together

In [ ]:
# Let's invoke our training loop to see the data the number of times we have set for epochs
best_valid_loss = float('inf') # Smaller loss is an indicator of a better model
for epoch in trange(epochs, desc="Epoch"):
  train_loss, train_acc = train(model, train_iter, loss_func, optimizer)
  valid_loss, valid_acc = evaluate(model, valid_iter, loss_func)

  # If our validation loss at the end of each epoch is better than our best, we want to save that model
  if valid_loss < best_valid_loss:
      best_valid_loss = valid_loss
      torch.save(model.state_dict(), 'my-sentiment-model.pt')

  print(f'\nEpoch: {epoch} | Train  Loss: {train_loss: .3f} | Train Acc: {train_acc*100:.2f}%')
  print(f'\nEpoch: {epoch} | Valid  Loss: {valid_loss: .3f} | Valid Acc: {valid_acc*100:.2f}%')

Epoch:   0%|          | 0/5 [00:00<?, ?it/s]

[0/1250] : Loss: 0.69
[100/1250] : Loss: 0.73
[200/1250] : Loss: 0.71
[300/1250] : Loss: 0.69
[400/1250] : Loss: 0.69
[500/1250] : Loss: 0.69
[600/1250] : Loss: 0.69
[700/1250] : Loss: 0.69
[800/1250] : Loss: 0.69
[900/1250] : Loss: 0.72
[1000/1250] : Loss: 0.69
[1100/1250] : Loss: 0.71
[1200/1250] : Loss: 0.69


Epoch:  20%|██        | 1/5 [00:05<00:20,  5.23s/it]


Epoch: 0 | Train  Loss:  0.701 | Train Acc: 49.89%

Epoch: 0 | Valid  Loss:  0.694 | Valid Acc: 49.62%
[0/1250] : Loss: 0.69
[100/1250] : Loss: 0.69
[200/1250] : Loss: 0.69
[300/1250] : Loss: 0.69
[400/1250] : Loss: 0.69
[500/1250] : Loss: 0.69
[600/1250] : Loss: 0.69
[700/1250] : Loss: 0.68
[800/1250] : Loss: 0.69
[900/1250] : Loss: 0.69
[1000/1250] : Loss: 0.69
[1100/1250] : Loss: 0.69
[1200/1250] : Loss: 0.69


Epoch:  40%|████      | 2/5 [00:10<00:15,  5.26s/it]

...


Epoch: 2 | Train  Loss:  0.710 | Train Acc: 50.30%

Epoch: 2 | Valid  Loss:  0.714 | Valid Acc: 50.32%
[0/1250] : Loss: 0.65
[100/1250] : Loss: 0.76
[200/1250] : Loss: 0.72
[300/1250] : Loss: 0.77
[400/1250] : Loss: 0.73
[500/1250] : Loss: 0.78
[600/1250] : Loss: 0.78
[700/1250] : Loss: 0.72
[800/1250] : Loss: 0.71
[900/1250] : Loss: 0.72
[1000/1250] : Loss: 0.72
[1100/1250] : Loss: 0.69
[1200/1250] : Loss: 0.79


Epoch:  80%|████████  | 4/5 [00:21<00:05,  5.26s/it]


Epoch: 3 | Train  Loss:  0.723 | Train Acc: 50.36%

Epoch: 3 | Valid  Loss:  0.724 | Valid Acc: 49.60%
[0/1250] : Loss: 0.65
[100/1250] : Loss: 0.72
[200/1250] : Loss: 0.68
[300/1250] : Loss: 0.69
[400/1250] : Loss: 0.71
[500/1250] : Loss: 0.71
[600/1250] : Loss: 0.69
[700/1250] : Loss: 0.69
[800/1250] : Loss: 0.69
[900/1250] : Loss: 0.69
[1000/1250] : Loss: 0.69
[1100/1250] : Loss: 0.69
[1200/1250] : Loss: 0.69


Epoch: 100%|██████████| 5/5 [00:26<00:00,  5.28s/it]


Epoch: 4 | Train  Loss:  0.701 | Train Acc: 50.03%

Epoch: 4 | Valid  Loss:  0.694 | Valid Acc: 49.84%


In [ ]:
# Finally let's load our best model and evaluate on our test data
model.load_state_dict(torch.load('my-sentiment-model.pt'))
test_loss, test_acc = evaluate(model, test_iter, loss_func)
print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')

Test Loss: 0.694 | Test Acc: 49.94%


Wow, these are not great results ...

How can we improve them?

Go back in this code and try to tweak the following parameters:
0. Think about the text pre-processing. How can you improve it to contain the "review" keywords that contain sentiment.
1. The input dimension, which determines the lenght of the reviews being considered
2. Selecting a different activation function from the list of activation functions in the model definition. (Attention, not all of them are useful for our problem)
3. Changing the number of epochs and learning rate, this is the number of times our model sees the data, and how confidentely it will update the model weight with respect to the losses.